In [30]:
import numpy as np 
import pandas as pd 

import re
import ssl
from urllib import request
import pymorphy3
import matplotlib.pyplot as plt
import seaborn as sns 

from catboost import CatBoostRegressor, Pool
import catboost as cb
from catboost import MetricVisualizer

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold

from scipy.sparse import csr_matrix
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
import spacy
from functools import lru_cache
from wordcloud import WordCloud
from tqdm.auto import trange

import os

from typing import Optional, Literal
from collections import Counter

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/maksimdegtarev/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [31]:
# Чтение данных
train = pd.read_parquet("train.parquet")
test = pd.read_parquet("test.parquet")
train.head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
0,185689,2024-09-27,Б/у,3000.0,Транспорт,Запчасти и аксессуары,Салон,1942218,1935418,Ручка АКПП mercedes w203 avangarde,Ручка рычаг акпп на Мерседес В203 mercedes w20...,185689.jpg,0.370,10.0,23.0,19.0
1,1914373,2024-11-07,Новое с биркой,5990.0,Личные вещи,"Одежда, обувь, аксессуары",Зимние куртки и пуховики,2164034,1753243,Пуховик Moncler голубой (52 размер),Объявление для заказа 📲\n\nАвито доставка 🚚\n\...,1914373.jpg,2.486,14.0,37.0,24.0
2,361626,2024-12-15,Новое,1200.0,Транспорт,Запчасти и аксессуары,Двигатель,621511,1233378,Запчасти на ford фокус1,"Опора задняя,двигатель w тигуан 1,4 150 б.у",361626.jpg,0.640,7.0,23.0,18.0
3,534927,2024-01-20,Б/у,13000.0,Электроника,"Игры, приставки и программы",Игровые приставки и аксессуары,998450,1082324,Ps3 cechc 08 скальпирована HEN(полностью испра...,"В комплекте приставка, провода(зарядка, питани...",534927.jpg,7.100,20.0,35.0,20.0
4,199043,2024-07-28,Отличное,300.0,Личные вещи,"Одежда, обувь, аксессуары","Джемперы, свитеры, кардиганы",528098,477834,Свитер трикотаж 44 р-р Reserved,Свитер Pull&Bear (пул энд бир)\n44 размер\nНад...,199043.jpg,0.400,7.0,23.0,11.0


### 1. Обработка пропусков

In [32]:
def get_missed_values_stat(df):
    missed_stat = df.isna().sum().sort_values(ascending=False).reset_index()
    missed_stat.columns = ['feature', 'NaN count']
    missed_stat['NaN share'] = missed_stat['NaN count'] / df.shape[0]
    return missed_stat[missed_stat['NaN count'] > 0]

def get_common_missed_data(df_train, df_test):
    missed_train = get_missed_values_stat(df_train)
    missed_test = get_missed_values_stat(df_test)

    missed_data = missed_train.merge(missed_test, how='outer', on='feature', suffixes=['_train', '_test'])
    
    return missed_data

In [33]:
# Проведем обработку пропусков
missed_data = get_common_missed_data(train, test)
missed_data

,feature,NaN count_train,NaN share_train,NaN count_test,NaN share_test
0,item_condition,16294,0.052073,3835,0.054572


#### Заменим пропуски в item_condition на "unknown"

In [34]:

train["item_condition"] = train["item_condition"].fillna("unknown").astype("str")
test["item_condition"] = test["item_condition"].fillna("unknown").astype("str")
missed_data = get_common_missed_data(train, test)
missed_data

,feature,NaN count_train,NaN share_train,NaN count_test,NaN share_test


In [35]:
seed = 42

# случайный сплит 90/10
train, val = train_test_split(
    train,
    test_size=0.1,
    random_state=seed,
    shuffle=True
)
print(f"train_size: {train.shape[0]}, val_size: {val.shape[0]}")

train_size: 281617, val_size: 31291


### 2. Первичный анализ данных (EDA) 

#### Анализ выбросов

In [36]:
targets = ["real_height", "real_length", "real_width", "real_weight"]
train[targets].describe().T

,count,mean,std,min,25%,50%,75%,max
real_height,281617.0,11.620215,8.325536,1.000,6.00,10.00,15.0,288.0
real_length,281617.0,33.191100,215.065406,1.000,22.00,32.00,40.0,111363.0
real_width,281617.0,23.502587,23.376426,1.000,16.00,22.00,30.0,11029.0
real_weight,281617.0,1.707787,20.997758,0.001,0.34,0.69,1.5,7090.0


#### Анализ таргетов указывает на наличие выбросов, подозрительно большие значения максимумов, особенно для real_length

In [37]:
train[targets].quantile([0.95, 0.99, 0.995, 0.999]).T

,0.950,0.990,0.995,0.999
real_height,29.0,36.0000,40.0,51.000000
real_length,56.0,99.0000,116.0,150.000000
real_width,44.0,52.0000,60.0,87.000000
real_weight,6.0,14.3084,18.0,24.416128


#### Квантили демонстрируют границы значительно меньшие чем максимальные значения, что подтвверждает предположение что объявления с максимальными значениями - выбросы. Рассмотрим примеры из данных, которые выходят за 0.999 квантиль

In [38]:
train[train["real_height"] > train["real_height"].quantile(0.999)].sort_values(by="real_height", ascending=False).head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
95915,1675102,2024-08-19,Б/у,100.0,Для дома и дачи,Посуда и товары для кухни,Товары для кухни,2178921,2391516,Хоз товары и тряпки,"Хоз товары, тряпки",1675102.jpg,15.81,288.0,437.0,321.0
34669,1088814,2024-03-28,Б/у,800.0,Электроника,Фототехника,Оборудование и аксессуары,422671,1949683,Цифровая фоторамка view sonic,"Модель VFМ1036w-11Е\nВ хорошем состоянии, почт...",1088814.jpg,1.81,153.0,333.0,253.0
197162,1661424,2024-11-12,Б/у,5400.0,Электроника,Оргтехника и расходники,"МФУ, копиры и сканеры",1093475,30370,Мфу лазерный Ricoh sp220 snw,"Хорошее рабочее состояние, без дефектов! Новый...",1661424.jpg,38.22,135.0,162.0,138.0
249173,1875634,2024-10-27,Б/у,9500.0,Для дома и дачи,Бытовая техника,Очистители воздуха,1153747,267046,Мойка воздуха Venta lw15,Увлажнитель - мойка воздуха venta lw15. В отли...,1875634.jpg,11.07,108.0,150.0,120.0
21117,1834023,2024-11-04,Новое с биркой,500.0,Личные вещи,"Одежда, обувь, аксессуары",Рюкзаки,1697859,626647,Новый классный рюкзак Гуси.Черный.Вышлю почтой,продам новый рюкзак Гуси.Вышлю почтой.,1834023.jpg,6.00,105.0,150.0,105.0


In [39]:
train[train["real_length"] > train["real_length"].quantile(0.999)].sort_values(by="real_length", ascending=False).head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
37315,1099165,2024-05-28,Б/у,2000.0,Электроника,Фототехника,Зеркальные фотоаппараты,1145987,891526,Тушка Canon 1000d,"Тушка canon 1000d, работает отлично, состояние...",1099165.jpg,0.850,16.0,111363.0,28.0
180033,216891,2024-01-11,unknown,370.0,Личные вещи,Красота и здоровье,Средства для волос,1345411,1467631,Schwarzkopf professional Кондиционер для волос,Кондиционер для волос PEPTIDE REPAIR rescue дл...,216891.jpg,0.560,1.0,21789.0,11029.0
77757,1686069,2024-08-23,Хорошее,30.0,Личные вещи,"Одежда, обувь, аксессуары",Сумки,534652,1523587,Отправлено Большая сумка женской обуви,"Продаю свою обувь: сапоги ботильоны, туфли и т...",1686069.jpg,12.900,30.0,7010.0,33.0
5264,618796,2024-01-21,Б/у,635.0,Хобби и отдых,Книги и журналы,Учебная литература,26525,1750451,"Учебник английского языка 1, 2, 3 класс технол...","Продаются учебники английского языка для 1, 2 ...",618796.jpg,1.075,3.0,3023.0,23.0
144014,683384,2024-01-13,Б/у,300.0,Хобби и отдых,Книги и журналы,Книги,492168,286503,"Кролик, который хочет уснуть",Книга в идеальным состоянии.\n\nВ профиле есть...,683384.jpg,0.476,6.0,2015.0,10.0


In [40]:
train[train["real_width"] > train["real_width"].quantile(0.999)].sort_values(by="real_width", ascending=False).head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
180033,216891,2024-01-11,unknown,370.0,Личные вещи,Красота и здоровье,Средства для волос,1345411,1467631,Schwarzkopf professional Кондиционер для волос,Кондиционер для волос PEPTIDE REPAIR rescue дл...,216891.jpg,0.56,1.0,21789.0,11029.0
95915,1675102,2024-08-19,Б/у,100.0,Для дома и дачи,Посуда и товары для кухни,Товары для кухни,2178921,2391516,Хоз товары и тряпки,"Хоз товары, тряпки",1675102.jpg,15.81,288.0,437.0,321.0
34669,1088814,2024-03-28,Б/у,800.0,Электроника,Фототехника,Оборудование и аксессуары,422671,1949683,Цифровая фоторамка view sonic,"Модель VFМ1036w-11Е\nВ хорошем состоянии, почт...",1088814.jpg,1.81,153.0,333.0,253.0
240024,1836483,2024-10-23,Б/у,5000.0,Личные вещи,Товары для детей и игрушки,Куклы и аксессуары,538886,348581,Дом мечты барби,"В идеальном состоянии, в комплекте куча мебели...",1836483.jpg,39.00,66.0,225.0,225.0
307037,1745944,2024-10-26,Б/у,540.0,Личные вещи,Товары для детей и игрушки,Постельные принадлежности,1400307,934415,Матрас для детей 120 60,Матрасик \nПользовались 3-4 Месяца. \n120*60,1745944.jpg,3.00,30.0,330.0,174.0


In [41]:
train[train["real_weight"] > train["real_weight"].quantile(0.999)].sort_values(by="real_weight", ascending=False).head()

,item_id,order_date,item_condition,item_price,category_name,subcategory_name,microcat_name,seller_id,buyer_id,title,description,image_name,real_weight,real_height,real_length,real_width
8337,139419,2024-07-10,Б/у,5000.0,Транспорт,Запчасти и аксессуары,Электрооборудование,487099,514504,Проводка Peugeot 206 1.6,Проводка Peugeot 206 1.6 2007г\n\n трёхдверный...,139419.jpg,7090.0,35.0,50.0,40.0
45302,1413997,2024-07-13,Б/у,1500.0,Электроника,Аудио и видео,"Акустика, колонки, сабвуферы",1186113,1371196,Аудиосистема Microlab M-200вт,Аудиосистема Microlab M-200вт. В рабочем состо...,1413997.jpg,4555.0,30.0,35.0,35.0
244562,148210,2024-01-31,Новое,3100.0,Для дома и дачи,Мебель и интерьер,Постельное бельё,2004742,2324580,Постельное бельё с одеялом евро,🌺Постельное белье с одеялом и простынью на ре...,148210.jpg,3250.0,13.0,50.0,40.0
232180,1475743,2024-06-18,Новое,750.0,Для дома и дачи,Ремонт и строительство,Водоочистка и фильтры,1841643,1258887,Жидкость для биотуалета Тhetford,Жидкость Thetford для биотуалета. для верхнег...,1475743.jpg,3000.0,10.0,10.0,10.0
232709,458921,2024-02-07,Б/у,200.0,Личные вещи,Детская одежда и обувь,Комбинезоны,1961529,2217273,Одежда пакетом 68 размер (забронировали),Отдам одежду для девочки\nБесплатно при самовы...,458921.jpg,2740.0,30.0,40.0,40.0


#### Визуальный анализ показывает что не все объявления являются мусором, есть валидные крупногабаритные объекты

#### Проанализируем распределение таргетов по категориям. Для анализа будем использовать соотношение хвостов для квантилей 0.99 и 0.999

In [42]:
cat_col = "subcategory_name"
targets = ["real_height", "real_length", "real_width", "real_weight"]

q_table = (
    train
    .groupby(cat_col)[targets]
    .quantile([0.99, 0.999])
    .unstack(level=1)
)

bad_subcats = set()

for t in targets:
    ratio = q_table[(t, 0.999)] / q_table[(t, 0.99)]

    print(f"\nПодозрительное соотношение хвостов для {t}")
    display(
        ratio
        .sort_values(ascending=False)
        .head(20)
    )

    # собираем subcategory, где ratio > 2
    bad_subcats.update(
        ratio[ratio > 2].index
    )

bad_subcats = list(bad_subcats)
bad_subcats


Подозрительное соотношение хвостов для real_height


subcategory_name
Мебель и интерьер               1.827750
Детская одежда и обувь          1.666667
Игры, приставки и программы     1.650857
Одежда, обувь, аксессуары       1.612903
Книги и журналы                 1.525500
Товары для детей и игрушки      1.463415
Посуда и товары для кухни       1.463415
Фототехника                     1.458333
Аудио и видео                   1.453650
Коллекционирование              1.452314
Часы и украшения                1.448750
Телефоны                        1.447000
Красота и здоровье              1.428571
Оргтехника и расходники         1.395349
Товары для компьютера           1.388889
Музыкальные инструменты         1.377000
Велосипеды                      1.345022
Охота и рыбалка                 1.336541
Бытовая техника                 1.333333
Планшеты и электронные книги    1.333333
dtype: float64


Подозрительное соотношение хвостов для real_length


subcategory_name
Телефоны                       2.168077
Книги и журналы                2.158947
Ноутбуки                       2.090050
Одежда, обувь, аксессуары      1.950000
Спорт и отдых                  1.898108
Товары для детей и игрушки     1.831883
Детская одежда и обувь         1.779661
Ремонт и строительство         1.731551
Красота и здоровье             1.714286
Коллекционирование             1.685271
Игры, приставки и программы    1.683367
Оргтехника и расходники        1.668407
Фототехника                    1.628750
Часы и украшения               1.618414
Товары для компьютера          1.607343
Бытовая техника                1.583961
Посуда и товары для кухни      1.531046
Аудио и видео                  1.523340
Мебель и интерьер              1.484463
Запчасти и аксессуары          1.392924
dtype: float64


Подозрительное соотношение хвостов для real_width


subcategory_name
Спорт и отдых                  1.960784
Одежда, обувь, аксессуары      1.953300
Мебель и интерьер              1.851852
Посуда и товары для кухни      1.788462
Детская одежда и обувь         1.772727
Товары для детей и игрушки     1.719950
Фототехника                    1.716667
Коллекционирование             1.675054
Игры, приставки и программы    1.646978
Бытовая техника                1.636444
Ремонт и строительство         1.587120
Книги и журналы                1.585568
Запчасти и аксессуары          1.495769
Товары для компьютера          1.493257
Красота и здоровье             1.458333
Оргтехника и расходники        1.416827
Аудио и видео                  1.403509
Телефоны                       1.363636
Часы и украшения               1.363636
Ноутбуки                       1.250000
dtype: float64


Подозрительное соотношение хвостов для real_weight


subcategory_name
Планшеты и электронные книги    2.700437
Одежда, обувь, аксессуары       2.695104
Часы и украшения                2.679646
Детская одежда и обувь          2.654082
Телефоны                        2.583245
Красота и здоровье              1.943593
Фототехника                     1.931133
Книги и журналы                 1.903146
Охота и рыбалка                 1.761254
Посуда и товары для кухни       1.753279
Ноутбуки                        1.695045
Мебель и интерьер               1.694582
Товары для детей и игрушки      1.633333
Спорт и отдых                   1.601708
Товары для компьютера           1.581768
Коллекционирование              1.481144
Бытовая техника                 1.474014
Игры, приставки и программы     1.464316
Оргтехника и расходники         1.407916
Настольные компьютеры           1.366655
dtype: float64

['Телефоны',
 'Детская одежда и обувь',
 'Одежда, обувь, аксессуары',
 'Книги и журналы',
 'Ноутбуки',
 'Часы и украшения',
 'Планшеты и электронные книги']

In [43]:
N_EXAMPLES = 10

for subcat in bad_subcats:
    df_sub = train[train[cat_col] == subcat]
    print(f"SUBCATEGORY: {subcat}")

    for t in targets:
        q99 = df_sub[t].quantile(0.99)
        q999 = df_sub[t].quantile(0.999)

        ratio = q999 / q99

        if ratio > 2:
            # показываем хвостовые примеры (те, кто > q99)
            tail_examples = (
                df_sub[df_sub[t] > q99]
                .sort_values(t, ascending=False)
                .head(N_EXAMPLES)
            )

            display(
                tail_examples[["item_id", "title", "description", t]]
            )

SUBCATEGORY: Телефоны


,item_id,title,description,real_length
23812,1694915,Радиостанция Intek M 790Plus,"Рация в рабочем состоянии, в комплекте антенна...",151.0
19700,1723908,"BQ 6035L Strike Power MAX, 2/32 ГБ",Разбили экран после пары месяцев как купили но...,150.0
155006,699049,Стационарная антенна для радиостанции,Антена для радиостанции си би. Длина 6м. Склад...,137.0
205523,1514237,Антенна radial A0 VHF,Бренд: РАДИАЛ 136-174 мгц VHF ✅\n\nКомплектаци...,120.0
970,1901920,Активная антенная система Yaesu atas-120А,Продаю автомобильную многодиапазонную (HF/VHF/...,105.0
207381,1670739,Базовая дипольная антенна D1 WHF Radial,Продам 4 петли на двухметровый диапазон. Прод...,83.0
106864,1006429,Антенна для рации anli новая,Anli AW-6 UHF Антенна автомобильная\nДиапазон ...,81.0
265828,1795740,Блок питания для радиостанции,Блок питания для радиостанции К-35 Alan.\n\nБл...,81.0
93858,1050205,Стойка для телефона,"Стойка для телефона, в разложенном состоянии 1...",80.0
191967,456303,"Fly IQ441 Radiance, 4 ГБ","Рабочий, но сенсор под замену! Аккумулятор раб...",78.0


,item_id,title,description,real_weight
99804,1867724,Коробка полная запчастей мобильных,коробка для алексея,24.750
139210,1439432,Планшеты много оптом рабочие и сломанные,"Планшеты - 1 ЛОТОМ ! ~ 200 шт. \nНа продажу, з...",24.500
299087,1733619,Кв трансивер UA1FA,Кв трансивер UA1FA пролежал 20 лет краска осы...,20.500
206734,698612,"Коробка с телефонами, электроникой и запчастями","Остаток от сервиса\nЕсть телефоны рабочие, нек...",16.000
171055,594608,"Телефон IP grandstream GXP2130, GXP2135",Продам БУ Телефоны IP GRANDSTREAM GXP2130-10шт...,15.000
266362,909239,500шт безпроводных наушников,Не рабочие на ремонт цена за все 500шт в налич...,15.000
159240,743971,Антенна для радиостанции Р163-10К,Комплект мачты с антенной для радиостанции Р16...,14.300
23812,1694915,Радиостанция Intek M 790Plus,"Рация в рабочем состоянии, в комплекте антенна...",13.197
166361,2014918,Армейский военно-полевой телефон та-57,Армейский военно-полевой телефонный аппарат ТА...,13.100
191695,1924695,"IP-телефоны Panasonic KX-NT511, Panasonic KX-N...",Комплект: \r\n6 IP-телефонов Panasonic KX-NT51...,12.032


SUBCATEGORY: Детская одежда и обувь


,item_id,title,description,real_weight
232709,458921,Одежда пакетом 68 размер (забронировали),Отдам одежду для девочки\nБесплатно при самовы...,2740.0
99404,380513,Вещи детские пакетом на 3-4 года,Вещи детские пакетом на 3-4 года . Цена за все...,1550.0
88777,754296,Новые вещи George Англия заказ 447 для Амины,🌈Заказ для Амины🌈\n\n✅️Смотрите наличие - kids...,800.0
307971,682310,Сапоги демисезонные детские Lassie 34 р,"Сапоги демисезонные Lassie, 34 размер. Материа...",780.0
107620,1038933,Сандали детские ecco 31,Сандали в отличном состоянии\nНо слились тольк...,350.0
258186,1457087,"Школьная юбка р. 140 Черная, Маленькая леди",Школьная юбка гофре для девочки. На рост 140 с...,320.0
292168,167792,Платье/ костюм Снежинка на 3 года,Красное платье/костюм Снежинки на Новогодний у...,246.0
221565,1948528,Коробка вещей для мальчика 68-80 размер,Коробка вещей для мальчика.,36.0
161435,1905827,"Детские вещи, обудь пакетоми","Отдам вещи , обувь детскую пакетами . \n6 паке...",25.0
255870,1893627,Чемодан вещей детских,Продам чемодан детских вещей на возврат 5 лет,25.0


...

,item_id,title,description,real_length
287110,821779,Ноутбук i5 3337/ gforce 740m (toshiba),"Строит новый ssd на 256 gb, windows 10. Работа...",1045.0
184179,1802697,Asus i7/12gb/ssd 240/Nvidia,Добро пожаловать в PRIME! \n\nНоутбук Asus в о...,140.0
257925,941861,"Ноутбук Irbis NB60, комплект, коробка, работает",Ноутбук в рабочем состоянии. От сети полностью...,138.0
255691,984368,Dell inspiron 7577 игровой,Хороший игровой ноутбук. Технически в идеально...,104.0
294617,812882,Коробки Macbook Pro Core i5 13 дюймов,Коробки от MacBook Pro Core i5 13 дюймов. Сост...,95.0
128501,1898705,"16"" huawei MateBook 16s cref-X i9-13900H",новый.\r\nна гарантии ДНС по всей РФ. \r\nполн...,83.0
184117,1639806,Ноутбук Lenovo Legion,Продается ноутбук Lenovo Legion + зарядное у...,80.0
13299,1160760,MSI GF63 RTX 3050/ I5 12450H / 16GB/ 144гц новый,"Игровой ноутбук, новый,запечатанный.\n\nБумажн...",80.0
189260,2092704,Asus Vivobook Pro 15 oled rtx3050,"ноутбук в очень хорошем состоянии, технически ...",80.0
292047,1015639,Ноутбук HP Pavilion DV6 7171er i7 ssd 256 8GB RAM,"Данный 15.6"" ноутбук с алюминиевой отделкой ко...",75.0


SUBCATEGORY: Часы и украшения


,item_id,title,description,real_weight
259063,488655,Часы Seiko Monster,Часы Seiko Monster 4- го поколения.\nМеханизм ...,300.000
296611,681860,Часы (нерабочие) и корпуса часов на запчасти,Закрылась мастерская часов. Продается одним ло...,16.500
269219,1171793,Настенные часы с зеркалом,Большие настенные часы. \n\nРАБОЧИЕ! \n\nЧасы ...,15.008
201834,1656147,Смарт часы лотом,Смарт часы около 80 шт лотом,13.900
209191,231977,Часы с боем Янтарь раритет,"Продаю совдеповские часы производство СССР, с ...",13.630
245800,1540114,Коробки для часов Longines 037-12,‼️БРОНЬ \n\nАВИТО ДОСТАВКА,13.200
300992,1553241,Коробки для часов Longines В12-2,Коробки для часов Longines \n\n‼️Бронь,13.200
275126,1626267,"Часы настенные, ГДР, из натур. кожи",Раритетные в очень хорошем сохране.Из натурал...,12.600
43677,2043669,"83х83 см. Большие Часы СССР,дерево,ручная работа","Часы настенные,ручная работа, Белые капли-это ...",12.600
292558,1181764,Колеса на Мотороллер,"Б/у колеса на мотороллер, в хорошем состоянии....",11.150


SUBCATEGORY: Планшеты и электронные книги


,item_id,title,description,real_weight
81421,1970582,"Аккумуляторы ICR 18650 (16,6 кг)",1) ICR 18650 сиреневый с подключением (5 шт.)\...,18.580
172787,1692120,Wacom Cintiq 16,"Гpафичecкий планшет в oтличнoм cостoянии, ред...",16.168
246876,714484,iPad 6 поколения,Покупался ребенку для просмотра мультиков.\n\n...,15.000
161139,1382298,Зарядная станция voyah free/zekker/lixiang,Новая станция\n\nПодходит для автомобилей Zekk...,13.454
103446,1230218,Чехлы на iPhone оптом,"🔸В связи с ликвидацией магазина, продаются ост...",13.350
58541,2004390,Аккумулятор повербанк для iPhone Magsafe,Новые отличные повербанки,12.600
55296,1593410,iPad air 4 98% акб,Приветствую вас 👋\nПродаю планшет Ipad air 4 6...,12.000
163230,1580389,Графический планшет XP-PEN Artist 24 pro,Продам графический планшет (монитор).\r\nПокуп...,11.200
16835,1002335,Графический планшет xp pen artist 24 pro,"Планшет в отличном состоянии, ни разу не испол...",10.200
156896,1483276,Графический монитор Xp pen Artist 24 Pro 2K QHD,"Состояние как новое ( сколов ,царапин, нет) П...",10.200


#### Заметим, что иногда в объявлении продается много товаров, что существенно влияет на вес, можно использовать как доп фичу если в поле description есть ключевые слова: шт, все, комплект, коробка, чемодан, много, оптом

#### Удалим данные из категорий в которых объявления больше всего похожи на выбросы

In [44]:
# чистим "Детская одежда и обувь"

train = train.loc[
    ~(
        (train["subcategory_name"] == "Детская одежда и обувь") &
        (train["real_weight"] > 36)
    )
].copy()


In [45]:
# чистим "Одежда, обувь, аксессуары"

q99 = (
    train.loc[train["subcategory_name"] == "Одежда, обувь, аксессуары", "real_weight"]
    .quantile(0.99)
)

train = train.loc[
    ~(
        (train["subcategory_name"] == "Одежда, обувь, аксессуары") &
        (train["real_weight"] > q99)
    )
].copy()


In [46]:
# чистим "Книги и журналы"

q99 = (
    train.loc[train["subcategory_name"] == "Книги и журналы", "real_length"]
    .quantile(0.99)
)

train = train.loc[
    ~(
        (train["subcategory_name"] == "Книги и журналы") &
        (train["real_length"] > q99)
    )
].copy()

In [47]:
# чистим "Часы и украшения"

train = train.loc[
    ~(
        (train["subcategory_name"] == "Часы и украшения") &
        (train["real_weight"] > 17)
    )
].copy()

#### Снова посмотрим статистики таргета после чистки

In [48]:
train[targets].describe().T

,count,mean,std,min,25%,50%,75%,max
real_height,280772.0,11.574445,8.266541,1.000,6.000,10.00,15.00,288.0
real_length,280772.0,33.077362,214.871513,1.000,22.000,32.00,40.00,111363.0
real_width,280772.0,23.452185,23.380571,1.000,16.000,21.00,30.00,11029.0
real_weight,280772.0,1.613904,19.267445,0.001,0.339,0.69,1.48,7090.0


#### Судя по максимальных значениям выбросы все равно остались. Сделаем клип по квантили внутри подкатегорий

In [49]:
q_clip = 0.999

for t in targets:
    thr = train.groupby("subcategory_name")[t].transform(lambda s: s.quantile(q_clip)) # каждая строка получает потолок, соответствующий своей подкатегории.
    train[t] = np.minimum(train[t], thr) # поэлементный минимум


In [50]:
train[targets].describe().T

,count,mean,std,min,25%,50%,75%,max
real_height,280772.0,11.559187,8.151298,1.000,6.000,10.00,15.00,73.11000
real_length,280772.0,32.519969,16.241165,1.000,22.000,32.00,40.00,210.69000
real_width,280772.0,23.396771,10.607179,1.000,16.000,21.00,30.00,105.00000
real_weight,280772.0,1.514239,2.571149,0.001,0.339,0.69,1.48,30.42534


#### Выглядит адекватно, перейдем к анализу выбросов с точки зрения минимальных значений

In [51]:
train[targets].quantile([0.0, 0.01, 0.05]).T

,0.00,0.01,0.05
real_height,1.000,1.000,1.0
real_length,1.000,7.000,10.0
real_width,1.000,4.000,10.0
real_weight,0.001,0.018,0.1


In [52]:
sentinel_values = {
    "real_height": 1,
    "real_length": 1,
    "real_width": 1,
    "real_weight": 0.001,
}

for t, v in sentinel_values.items():
    print(t, (train[t] == v).mean())


real_height 0.12892667359993162
real_length 0.0012501246563047597
real_width 0.003910646360748222
real_weight 0.006172267889960537


#### Анализ показал, что почти 13 % объявлений имеют высоту 1 мм, скорее всего это заглушка и указана вместо реальной высоты. Такие значения могут сильно повредить регрессии, поэтому заменим их на медианные значения по микрокатегории

In [ ]:
def impute_stub_by_micro_median(
    df: pd.DataFrame,
    target: str,
    stub_value: float,
    micro_col: str = "microcat_name",
    min_count: int = 20,
) -> pd.DataFrame:
    """
    Заменяет заглушки target == stub_value на медиану target
    по микрокатегории, если в ней более min_count данных,
    иначе на глобальную медиану.
    """

    df = df.copy()

    # глобальная медиана по нормальным значениям
    global_median = df.loc[df[target] != stub_value, target].median()

    # считаем медианы и размеры групп без заглушек
    stats = (
        df.loc[df[target] != stub_value]
        .groupby(micro_col)[target]
        .agg(median="median", cnt="size")
    )

    # маска заглушек
    mask_stub = df[target] == stub_value

    # замена значений
    df.loc[mask_stub, target] = (
        df.loc[mask_stub, micro_col]
        .map(
            lambda x: (
                stats.loc[x, "median"]
                if (x in stats.index and stats.loc[x, "cnt"] >= min_count)
                else global_median
            )
        )
    )

    return df


In [ ]:
# заменяем значения у которых real_height == 1
train = impute_stub_by_micro_median(
    df=train,
    target="real_height",
    stub_value=1.0,
)

In [63]:
# заменяем значения у которых real_length == 1
train = impute_stub_by_micro_median(
    df=train,
    target="real_length",
    stub_value=1.0,
)

In [64]:
# заменяем значения у которых real_width == 1
train = impute_stub_by_micro_median(
    df=train,
    target="real_width",
    stub_value=1.0,
)

#### Проанализируем объявления у которых слишком малый вес, для этого посмотрим у каких категорий есть резкий скачок от нижних значений к нормальным

In [58]:
cat_col = "subcategory_name"
target = "real_weight"

q_table = (
    train
    .groupby(cat_col)[targets]
    .quantile([0.01, 0.05])
    .unstack(level=1)
)

bad_subcats = set()

for t in targets:
    ratio = q_table[(t, 0.05)] / q_table[(t, 0.01)]
    
    display(
        ratio
        .sort_values(ascending=False)
        .head(15)
    )

    bad_subcats.update(
        ratio[ratio > 10].index
    )

bad_subcats = list(bad_subcats)
bad_subcats


subcategory_name
Товары для компьютера         2.000000
Охота и рыбалка               2.000000
Оргтехника и расходники       2.000000
Музыкальные инструменты       2.000000
Ремонт и строительство        1.700000
Настольные компьютеры         1.682243
Спорт и отдых                 1.666667
Фототехника                   1.666667
Товары для детей и игрушки    1.666667
Одежда, обувь, аксессуары     1.666667
Мебель и интерьер             1.666667
Бытовая техника               1.500000
Часы и украшения              1.500000
Коллекционирование            1.500000
Телефоны                      1.500000
dtype: float64

subcategory_name
Телефоны                        10.000000
Книги и журналы                  5.000000
Игры, приставки и программы      3.666667
Часы и украшения                 3.333333
Коллекционирование               3.333333
Детская одежда и обувь           3.200000
Ноутбуки                         3.000000
Планшеты и электронные книги     2.500000
Товары для компьютера            2.500000
Настольные компьютеры            1.875000
Велосипеды                       1.841621
Товары для детей и игрушки       1.714286
Мебель и интерьер                1.555556
Одежда, обувь, аксессуары        1.500000
Посуда и товары для кухни        1.395000
dtype: float64

subcategory_name
Книги и журналы                 10.000000
Часы и украшения                 7.000000
Телефоны                         6.000000
Игры, приставки и программы      5.000000
Товары для компьютера            4.500000
Планшеты и электронные книги     4.000000
Коллекционирование               4.000000
Детская одежда и обувь           3.333333
Велосипеды                       2.666667
Товары для детей и игрушки       2.500000
Музыкальные инструменты          2.368421
Спорт и отдых                    2.000000
Посуда и товары для кухни        2.000000
Ноутбуки                         2.000000
Мебель и интерьер                2.000000
dtype: float64

subcategory_name
Детская одежда и обувь        120.000000
Мебель и интерьер              90.000000
Книги и журналы                86.206897
Посуда и товары для кухни      63.926941
Товары для детей и игрушки     50.000000
Бытовая техника                32.600000
Ремонт и строительство         30.000000
Оргтехника и расходники         8.453085
Одежда, обувь, аксессуары       5.000000
Охота и рыбалка                 4.357298
Музыкальные инструменты         4.032258
Товары для компьютера           3.957500
Коллекционирование              3.808487
Спорт и отдых                   3.716667
Ноутбуки                        3.702693
dtype: float64

['Товары для детей и игрушки',
 'Детская одежда и обувь',
 'Мебель и интерьер',
 'Книги и журналы',
 'Бытовая техника',
 'Ремонт и строительство',
 'Посуда и товары для кухни']

#### Категории в которых отношение q05/q01 > 10 скорей всего содержат заглушки по весам, либо неправдоподоно мелкие веса

In [59]:
N_EXAMPLES = 10

for subcat in bad_subcats:
    df_sub = train[train[cat_col] == subcat]
    print(f"SUBCATEGORY: {subcat}")

    for t in targets:
        q01 = df_sub[t].quantile(0.01)
        q05 = df_sub[t].quantile(0.05)

        ratio = q05 / q01

        if ratio > 20:
            tail_examples = (
                df_sub[df_sub[t] < q05]
                .sort_values(t)
                .head(N_EXAMPLES)
            )

            display(
                tail_examples[["item_id", "title", "description", t]]
            )

SUBCATEGORY: Товары для детей и игрушки


,item_id,title,description,real_weight
189154,1629629,Автокресло для Baby born,"Автокресло для куколки, размером Baby born (43...",0.001
151258,938902,Бластеры с мягкими пулями,"2 бластера с мягкими пулями,1 синий бластер, ...",0.001
251893,1728803,"Конструктор ""Полесье""","Продам конструктор ""Полесье"" в хорошем состоян...",0.001
111834,270795,Пазл карта мира Castorland maxi,Maxi пазл Castorland «Карта мира»\nОдна деталь...,0.001
257684,1753662,Зап части доя коляски,Зап части для коляски,0.001
282854,1778904,Вещи для куклы Реборн девочка 50 56р,Комплект вещей для куклы Реборн девочка 50 56р...,0.001
109805,232619,Лежак детский с антимоскитной сеткой,"Очень удобный лежак, можно ходить в гости, на ...",0.001
199447,1767328,Оранжевая Синяя Зеленая книга сказок 3шт,Состояние новых. Не пользовались\r\nЦена за все,0.001
164753,1361472,Пазл детский djeco 24 детали улитка,Пазл Djeco 24 детали улитка. Оригинал. Для дет...,0.001
171068,1301041,"Горшок детский складной, дорожный","Многофункциональный, компактный, легкий горшок.",0.001


SUBCATEGORY: Детская одежда и обувь


,item_id,title,description,real_weight
250961,1890923,Зимняя куртка 128 футурино,Новая зимняя парка для мальчика рост 128\n\nУт...,0.001
180937,1824465,Комбинезон детский HM,"Детский комбинезон HM, холодный демисезон/евро...",0.001
271610,1572424,Детские резиновые сапоги 36-37,"сапожки почти новые, одеты пару раз.\nМожно но...",0.001
248266,1423728,Сандали на мальчика размер 35,"Продам сандали, состояние хорошее. Размер 35",0.001
270103,1616455,Босоножки для девочки 30 размер tapiboo,Абсолютно новые \nИз за большого количества об...,0.001
8415,1170721,Сандалии детские для мальчиков 27,"Поносили в сад несколько месяцев,состояние иде...",0.001
86381,309374,Демисезонный костюм для девочки 122,Новые демисезонные костюмы фирмы Крош. Плотная...,0.001
21250,8085,Колготки пакетом на девочку 2-3 года,Пакетом на девочку колготки 2-3 года,0.001
159368,1377884,Сланцы crocs (размер 27),Сланцы в отличном состоянии.,0.001
147843,1812853,Зимние ботинки 30р,Зимние ботинки Tiflani для девочки 30 размер в...,0.001


SUBCATEGORY: Мебель и интерьер


...

,item_id,title,description,real_weight
299985,1271264,Чайник электрический BQ,✅Чайник на гарантии до октября 2024года\n✅Функ...,0.001
4832,1988491,Ручной отпариватель на доставке,"Не пользовалась, но упаковка утеряна. SwissHom...",0.001
195414,1649994,Аккумулятор dyson v7,Почти не рабочий держит на максимуме 10 секунд...,0.001
283926,240505,Гриль GFgril новый,"Фото 1 - «gf-065» - 2700р. \n\nНовый, в коробк...",0.001
140520,1349337,Новая элекрическая бритва,"Новая электро бритва Timberk \nБрала за 2000, ...",0.001
156055,333926,"Овощерезка, измельчитель, бу","Состояние отличное, стоит, продаем потому что ...",0.001
87981,1512743,Выпрямитель для волос dewal ceramic base,Утюжок - выпрямитель для волос dewal ceramic b...,0.001
89900,88515,Утюг Polaris PIR 2494K беспроводной,"БРОНЬ\r\nУтюг Polaris PIR 2494 Cordless, limit...",0.001
262604,1910898,Соковыжималка электрическая,Соковыжималка электрическая Tefal Nectalia. Пр...,0.001
229101,1898767,Комплект от пылесоса,Для доставки\r\n1. Канистра для детергентов\r\...,0.001


SUBCATEGORY: Ремонт и строительство


,item_id,title,description,real_weight
281666,1028658,Шланг для мойки высокого давления 10 метров,Шланг высокого давления 10 метров \n\nГайка М2...,0.001
303386,169817,Гравёр Bosch GRO 12v-35,Аккумуляторный гравер Bosch GRO 12V-35 \n\nКом...,0.001
197569,1122788,Газ лифты б/у,Газ лифты б/у. 200 за один газлифт. (остались...,0.001
252248,1583332,Смеситель для раковины бу,Б/у совсем не много. Тяжелый качественный. Ита...,0.001
303723,523716,Насадка-заклепочник зубр на шуруповерт М3 - М8,асадка-заклепочник ЗУБР Профессионал НР-М8 М3 ...,0.001
245380,410727,Розетка выключатель usb,"розетки выключатели usb\nкабель кпс 1х2х0,5\nв...",0.001
218753,706505,Линейный подшипник LM8UU,Новые 10 шт. Продаю так как пришли не того раз...,0.001
90443,438986,Гофра 150 алюминевая,150 гофра отличного качества 300р,0.001
114164,1118469,Прибор измерительный Ц435,"Мультиметр СССР, работоспособность неизвестна.",0.001
85949,1355259,Поливочный шланг новый,"Поливочный шланг 22,5 метра, в наличии 25 шт.",0.001


SUBCATEGORY: Посуда и товары для кухни


,item_id,title,description,real_weight
306192,1803718,Чайник заварочный 1 литр,Есть небольшое повреждение: ручка внизу оплавл...,0.001
189807,1157760,Магнитный держатель подставка для ножей,"Новая подставка для ножей, была в комплекте, н...",0.001
269240,1577564,Маслёнка масленица фарфор,лежит без дела. \nновая . \nпересыл .\nотдам б...,0.001
8898,1870193,Гейзерная кофеварка индукция 450мл,"Покупали в подарок, подарить не получилось.\n\...",0.001
8272,1157434,Овощерезка 5 в 1 mlyncek,Ручной измельчитель MLYNCEK 5 в 1 - ручная тер...,0.001
236002,854832,Кофе в капсулах для кофемашины,продаю две банки сразу - 300 руб самовывоз \nс...,0.001
154434,270350,"Подставка для стаканов, бутылок","Подставка. Стаканы в комплект не входят, прост...",0.001
888,2049875,Вилки нержавейка СССР 8 штук,Вилки ссср нержавейка 8 шт,0.001
301662,1558919,Форма для выпечки кексов кулича антипригарная,Форма для выпечки антипригарная кексов кулича ...,0.001
101245,1457538,Amway чистящее средство Loc,Подходит для очищения всех непористых поверхно...,0.001


#### Анализ объявлений показывает что вес 0.001 - тоже похоже на заглушку, заменим вес для этих объявлений на средний по микрокатегории

In [60]:
train = impute_stub_by_micro_median(
    df=train,
    target="real_weight",
    stub_value=0.001
)

### 3. Добавление новых фичей

In [67]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 280772 entries, 225794 to 121958
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   item_id           280772 non-null  int64  
 1   order_date        280772 non-null  object 
 2   item_condition    280772 non-null  object 
 3   item_price        280772 non-null  float64
 4   category_name     280772 non-null  object 
 5   subcategory_name  280772 non-null  object 
 6   microcat_name     280772 non-null  object 
 7   seller_id         280772 non-null  int64  
 8   buyer_id          280772 non-null  int64  
 9   title             280772 non-null  object 
 10  description       280772 non-null  object 
 11  image_name        280772 non-null  object 
 12  real_weight       280772 non-null  float64
 13  real_height       280772 non-null  float64
 14  real_length       280772 non-null  float64
 15  real_width        280772 non-null  float64
dtypes: float64(5), int64

In [68]:
feat_extra_cat = []
feat_extra_num = []

In [69]:
# объединим в одну фичу title + description

def add_text_join(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text_join"] = (
        df["title"] + " " +
        df["description"].fillna("")
    )
    return df

train = add_text_join(train)
val = add_text_join(val)
test = add_text_join(test)

In [70]:
# Добавим фичу доля заглавных букв в title
def caps_ratio(s: str) -> float:
    """доля заглавных букв в тексте,
    рассчитанная как отношение количества
    заглавных букв к общему количеству букв"""
    if not s:
        return 0.0

    letters = sum(ch.isalpha() for ch in s)
    if letters == 0:
        return 0.0

    caps = sum(ch.isupper() for ch in s)
    return caps / letters

train["caps_ratio"] = train["title"].apply(lambda x: caps_ratio(x))
val["caps_ratio"] = val["title"].apply(lambda x: caps_ratio(x))
test["caps_ratio"] = test["title"].apply(lambda x: caps_ratio(x))

feat_extra_num.append("caps_ratio")

In [71]:
# Добавим фичу доля букв и цифр в title 

def alnum_ratio(s: str) -> float:
    """доля букв и цифр в тексте,
    рассчитанная как отношение количества
    букв и цифр к общему количеству символов с пробелами"""
    if not s:
        return 0.0

    total = len(s)
    if total == 0:
        return 0.0

    alnum = sum(ch.isalnum() for ch in s)
    return alnum / total

train["alnum_ratio"] = train["title"].apply(lambda x: alnum_ratio(x))
val["alnum_ratio"] = val["title"].apply(lambda x: alnum_ratio(x))
test["alnum_ratio"] = test["title"].apply(lambda x: alnum_ratio(x))

feat_extra_num.append("alnum_ratio")

In [72]:
# добавим фичу количество символов в title
train["text_len"] = train["title"].apply(lambda x: len(x))
val["text_len"] = val["title"].apply(lambda x: len(x))
test["text_len"] = test["title"].apply(lambda x: len(x))

feat_extra_num.append("text_len")

In [73]:
# log количество слов title
train["log_words_cnt"] = train["title"].apply(lambda x: np.log1p(len(x.split())))
val["log_words_cnt"] = val["title"].apply(lambda x: np.log1p(len(x.split())))
test["log_words_cnt"] = test["title"].apply(lambda x: np.log1p(len(x.split())))

feat_extra_num.append("log_words_cnt")

In [74]:
# сделаем фичу - флаг, что в объявлении множество товаров

pattern = re.compile(
    r"(?:\bшт\b|\d+\s*шт|\bштук\b|\bкомплект\b|\bчемодан\b|\bнабор\b|\bоптом\b)",
    flags=re.IGNORECASE
)

train["has_bundle_words"] = (
    train["text_join"]
    .fillna("")
    .str.lower()
    .apply(lambda x: int(bool(pattern.search(x))))
)

val["has_bundle_words"] = (
    val["text_join"]
    .fillna("")
    .str.lower()
    .apply(lambda x: int(bool(pattern.search(x))))
)

test["has_bundle_words"] = (
    test["text_join"]
    .fillna("")
    .str.lower()
    .apply(lambda x: int(bool(pattern.search(x))))
)
 
feat_extra_cat.append("has_bundle_words")

In [75]:
# сделаем фичу - флаг, что в объявлении есть единицы измерения

train["has_size"] = train["text_join"].str.contains(
    r"(?:\bмм\b|\bсм\b|\bм\b|\bметр(?:а|ов)?\b|\d+(?:[.,]\d+)?\s*(?:мм|см|м)\b)",
    case=False,
    regex=True
).astype(int)

val["has_size"] = val["text_join"].str.contains(
    r"(?:\bмм\b|\bсм\b|\bм\b|\bметр(?:а|ов)?\b|\d+(?:[.,]\d+)?\s*(?:мм|см|м)\b)",
    case=False,
    regex=True
).astype(int)

test["has_size"] = test["text_join"].str.contains(
    r"(?:\bмм\b|\bсм\b|\bм\b|\bметр(?:а|ов)?\b|\d+(?:[.,]\d+)?\s*(?:мм|см|м)\b)",
    case=False,
    regex=True
).astype(int)

feat_extra_cat.append("has_size")

In [76]:
# добавим фичи месяц заказа order_month и день недели order_dow
dt_train = pd.to_datetime(train["order_date"], errors="coerce")
train["order_month"] = dt_train.dt.month.astype("Int64")
train["order_dow"] = dt_train.dt.dayofweek.astype("Int64") 

dt_val = pd.to_datetime(val["order_date"], errors="coerce")
val["order_month"] = dt_val.dt.month.astype("Int64")
val["order_dow"] = dt_val.dt.dayofweek.astype("Int64") 

dt_test = pd.to_datetime(test["order_date"], errors="coerce")
test["order_month"] = dt_test.dt.month.astype("Int64")
test["order_dow"] = dt_test.dt.dayofweek.astype("Int64") 

feat_extra_cat.append("order_month")
feat_extra_cat.append("order_dow")


In [77]:
# добавим фичу log_item_price
train["log_item_price"] = np.log1p(train["item_price"])
val["log_item_price"] = np.log1p(val["item_price"])
test["log_item_price"] = np.log1p(test["item_price"])

feat_extra_num.append("log_item_price")

### 4. Генерация мета фичей на out of folds preds (OOF)

#### 4.1 Ridge на char-tf-idf векторах title clean

#### 4.1.1 Очистка title для tf-idf

In [78]:
russian_stopwords = set(stopwords.words("russian"))

def simple_clean(s: str) -> str:
    """
    Функция для простой чистки текстов
    """
    if not s:
        return ""
    
    s = s.lower().replace("ё", "е")
    s = re.sub(r"[^a-zа-я0-9\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    toks = s.split(" ")
    out = []

    for t in toks:
        if len(t) < 2 or t in russian_stopwords:
            continue
        out.append(t)

    return " ".join(out)

train["title_clean"] = train["title"].apply(lambda x: simple_clean(x))
train["text_join_clean"] = train["text_join"].apply(lambda x: simple_clean(x))

val["title_clean"] = val["title"].apply(lambda x: simple_clean(x))
val["text_join_clean"] = val["text_join"].apply(lambda x: simple_clean(x))

test["title_clean"] = test["title"].apply(lambda x: simple_clean(x))
test["text_join_clean"] = test["text_join"].apply(lambda x: simple_clean(x))


#### 4.1.2 Сделаем OOF предсказания таргетов на title_clean char-tf-idf векторах

In [79]:
targets = ["real_height", "real_length", "real_width", "real_weight"]

def oof_ridge_tfidf_multi_log(
    train_df: pd.DataFrame,
    holdout_df: pd.DataFrame,
    test_df: pd.DataFrame,
    text_col: str,
    targets: list[str],
    n_splits: int = 5,
    seed: int = 42,
    # tfidf params
    analyser="word",
    ngram_range=(1, 2),
    min_df: int = 2,
    max_features: int = 200_000,
    sublinear_tf: bool = True,
    # ridge param
    alpha: float = 10.0,
):  
    X_text = train_df[text_col].values
    X_text_test = test_df[text_col].values
    X_text_hold = holdout_df[text_col].values

    # словари для хранения OOF_preds по каждому таргету
    oof_log = {t: np.zeros(len(train_df), dtype=np.float32) for t in targets}
    test_log_sum = {t: np.zeros(len(test_df), dtype=np.float32) for t in targets}
    hold_log_sum = {t: np.zeros(len(holdout_df), dtype=np.float32) for t in targets}

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # обучение по фолдам
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_text), 1):
        X_tr_raw = X_text[tr_idx]
        X_val_raw = X_text[val_idx]

        vec = TfidfVectorizer(
            analyzer=analyser,
            ngram_range=ngram_range,
            min_df=min_df,
            max_features=max_features,
            sublinear_tf=sublinear_tf, # заменяет tf на 1 + log(tf)
        )
        X_tr = vec.fit_transform(X_tr_raw)
        X_val = vec.transform(X_val_raw)
        X_test = vec.transform(X_text_test)
        X_h = vec.transform(X_text_hold)

        for t in targets:
            y = train_df[t].values
            y_tr_log = np.log1p(y[tr_idx])

            model = Ridge(alpha=alpha)
            model.fit(X_tr, y_tr_log)

            # val fold oof preds
            oof_log[t][val_idx] = model.predict(X_val).astype(np.float32)

            # holdout df preds (среднее по фолдам)
            hold_log_sum[t] += model.predict(X_h).astype(np.float32)

            # test preds (среднее по фолдам)
            test_log_sum[t] += model.predict(X_test).astype(np.float32)
        
        print(f"fold {fold}/{n_splits} done")

    # OOF dataframe
    oof_df = pd.DataFrame(
        {f"oof_ridge_{analyser}_{text_col}_{t}_log": oof_log[t] for t in targets},
        index=train_df.index
    )

    holdout_pred_df = pd.DataFrame(
        {f"oof_ridge_{analyser}_{text_col}_{t}_log": (hold_log_sum[t] / n_splits) for t in targets},
        index=holdout_df.index
    )

    test_pred_df = pd.DataFrame(
        {f"oof_ridge_{analyser}_{text_col}_{t}_log": (test_log_sum[t] / n_splits) for t in targets},
        index=test_df.index
    )

    # log MAE
    scores = []
    for t in targets:
        y_true_log = np.log1p(train_df[t].values)
        y_pred_log = oof_log[t]
        mae_log = mean_absolute_error(y_true_log, y_pred_log)
        scores.append({"target": t, "mae_log": mae_log})
    
    scores_df = pd.DataFrame(scores)
    scores_df.loc["mean"] = ["mean", scores_df["mae_log"].mean()]

    return oof_df, holdout_pred_df, test_pred_df, scores_df


train_oof_char_title_log, val_oof_char_title_log, test_oof_char_title_log, scores_char = oof_ridge_tfidf_multi_log(
    train_df=train,
    holdout_df=val,
    test_df=test,
    text_col="title_clean",
    targets=targets,
    n_splits=5,
    seed=42,
    analyser="char",
    ngram_range=(2, 4),
    min_df=2,
    max_features=200_000,
    sublinear_tf=True,
    alpha=10.0,
)

train = pd.concat([train, train_oof_char_title_log], axis=1).copy()
val = pd.concat([val, val_oof_char_title_log], axis=1).copy()
test = pd.concat([test, test_oof_char_title_log], axis=1).copy()

display(scores_char)
train_oof_char_title_log.head()

fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done


,target,mae_log
0,real_height,0.332742
1,real_length,0.309066
2,real_width,0.309723
3,real_weight,0.318583
mean,mean,0.317528


,oof_ridge_char_title_clean_real_height_log,oof_ridge_char_title_clean_real_length_log,oof_ridge_char_title_clean_real_width_log,oof_ridge_char_title_clean_real_weight_log
225794,2.336086,3.295370,3.042905,0.666144
127330,2.361759,3.468079,2.957355,0.564329
76364,2.322227,3.221870,2.993950,0.318945
79315,2.589854,3.477914,3.222682,0.809728
285898,2.645033,3.561268,3.277112,0.755481


In [80]:
# сохраним названия фичей
feat_oof_ridge_tfidf_title_clean = train_oof_char_title_log.columns.tolist()

#### 4.1.3 Сгенерим еще extra фичи на основе ridge char title clean oof preds log

In [81]:
# сделаем отдельный список для названий экстра мета фичей
feat_meta_proxy = []

In [82]:
# log отношения размеров
# oof_len_log - oof_height_log
train["oof_ridge_char_title_clean_log_ratio_l_h"] = train["oof_ridge_char_title_clean_real_length_log"] - train["oof_ridge_char_title_clean_real_height_log"]
val["oof_ridge_char_title_clean_log_ratio_l_h"] = val["oof_ridge_char_title_clean_real_length_log"] - val["oof_ridge_char_title_clean_real_height_log"]
test["oof_ridge_char_title_clean_log_ratio_l_h"] = test["oof_ridge_char_title_clean_real_length_log"] - test["oof_ridge_char_title_clean_real_height_log"]

# #oof_width_log - oof_height_log
train["oof_ridge_char_title_clean_log_ratio_w_h"] = train["oof_ridge_char_title_clean_real_width_log"] - train["oof_ridge_char_title_clean_real_height_log"]
val["oof_ridge_char_title_clean_log_ratio_w_h"] = val["oof_ridge_char_title_clean_real_width_log"] - val["oof_ridge_char_title_clean_real_height_log"]
test["oof_ridge_char_title_clean_log_ratio_w_h"] = test["oof_ridge_char_title_clean_real_width_log"] - test["oof_ridge_char_title_clean_real_height_log"]

In [83]:
feat_meta_proxy.append("oof_ridge_char_title_clean_log_ratio_l_h")
feat_meta_proxy.append("oof_ridge_char_title_clean_log_ratio_w_h")

In [84]:
# порядок величин
train["oof_ridge_char_title_clean_is_len_max"] = (
    train["oof_ridge_char_title_clean_real_length_log"]
    > train[[
        "oof_ridge_char_title_clean_real_width_log",
        "oof_ridge_char_title_clean_real_height_log"
    ]].max(axis=1)
).astype(int)

val["oof_ridge_char_title_clean_is_len_max"] = (
    val["oof_ridge_char_title_clean_real_length_log"]
    > val[[
        "oof_ridge_char_title_clean_real_width_log",
        "oof_ridge_char_title_clean_real_height_log"
    ]].max(axis=1)
).astype(int)

test["oof_ridge_char_title_clean_is_len_max"] = (
    test["oof_ridge_char_title_clean_real_length_log"]
    > test[[
        "oof_ridge_char_title_clean_real_width_log",
        "oof_ridge_char_title_clean_real_height_log"
    ]].max(axis=1)
).astype(int)

In [85]:
feat_meta_proxy.append("oof_ridge_char_title_clean_is_len_max")

In [86]:
# стандарное отклонение
train["oof_ridge_char_title_clean_spread_log"] = train[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log"
]].std(axis=1)

val["oof_ridge_char_title_clean_spread_log"] = val[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log"
]].std(axis=1)

test["oof_ridge_char_title_clean_spread_log"] = test[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log"
]].std(axis=1)

In [87]:
feat_meta_proxy.append("oof_ridge_char_title_clean_spread_log")

In [88]:
# proxy объема
# oof_volume_log ≈ oof_h_log + oof_w_log + oof_l_log
train["oof_ridge_char_title_clean_volume_log"] = train[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log",
]].sum(axis=1)

val["oof_ridge_char_title_clean_volume_log"] = val[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log",
]].sum(axis=1)

test["oof_ridge_char_title_clean_volume_log"] = test[[
    "oof_ridge_char_title_clean_real_width_log",
    "oof_ridge_char_title_clean_real_height_log",
    "oof_ridge_char_title_clean_real_length_log",
]].sum(axis=1)

In [89]:
feat_meta_proxy.append("oof_ridge_char_title_clean_volume_log")

In [90]:
# proxy плотности
train["oof_ridge_char_title_clean_log_density"] = (
    train["oof_ridge_char_title_clean_real_weight_log"]
    - train["oof_ridge_char_title_clean_volume_log"]
)

val["oof_ridge_char_title_clean_log_density"] = (
    val["oof_ridge_char_title_clean_real_weight_log"]
    - val["oof_ridge_char_title_clean_volume_log"]
)

test["oof_ridge_char_title_clean_log_density"] = (
    test["oof_ridge_char_title_clean_real_weight_log"]
    - test["oof_ridge_char_title_clean_volume_log"]
)

In [91]:
feat_meta_proxy.append("oof_ridge_char_title_clean_log_density")

#### 4.2 Ridge на word-tf-idf векторах title_clean

In [92]:
train_oof_word_title_log, val_oof_word_title_log, test_oof_word_title_log, scores_word = oof_ridge_tfidf_multi_log(
    train_df=train,
    holdout_df=val,
    test_df=test,
    text_col="title_clean",
    targets=targets,
    n_splits=5,
    seed=42,
    analyser="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=200_000,
    sublinear_tf=True,
    alpha=10.0,
)

train = pd.concat([train, train_oof_word_title_log], axis=1)
val = pd.concat([val, val_oof_word_title_log], axis=1)
test = pd.concat([test, test_oof_word_title_log], axis=1)

display(scores_word)
train_oof_word_title_log.head()

fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done


,target,mae_log
0,real_height,0.329450
1,real_length,0.304819
2,real_width,0.306366
3,real_weight,0.307825
mean,mean,0.312115


,oof_ridge_word_title_clean_real_height_log,oof_ridge_word_title_clean_real_length_log,oof_ridge_word_title_clean_real_width_log,oof_ridge_word_title_clean_real_weight_log
225794,2.361991,3.252728,2.962073,0.614894
127330,2.518994,3.337769,3.033105,0.654446
76364,2.299138,3.185628,2.959938,0.311360
79315,2.487179,3.468314,3.224682,0.686956
285898,2.598393,3.484463,3.252125,0.631177


In [93]:
feat_oof_ridge_word_title_clean = train_oof_word_title_log.columns.tolist()

#### 4.3 Ridge на word-tf-idf векторах text_join_clean (title + description)

In [94]:
train_oof_word_text_join_log, val_oof_word_text_join_log, test_oof_word_text_join_log, scores_word_text_join = oof_ridge_tfidf_multi_log(
    train_df=train,
    holdout_df=val,
    test_df=test,
    text_col="text_join_clean",
    targets=targets,
    n_splits=5,
    seed=42,
    analyser="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=200_000,
    sublinear_tf=True,
    alpha=10.0,
)

train = pd.concat([train, train_oof_word_text_join_log], axis=1)
val = pd.concat([val, val_oof_word_text_join_log], axis=1)
test = pd.concat([test, test_oof_word_text_join_log], axis=1)

display(scores_word_text_join)
train_oof_word_text_join_log.head()

fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done


,target,mae_log
0,real_height,0.327438
1,real_length,0.305269
2,real_width,0.306539
3,real_weight,0.307008
mean,mean,0.311563


,oof_ridge_word_text_join_clean_real_height_log,oof_ridge_word_text_join_clean_real_length_log,oof_ridge_word_text_join_clean_real_width_log,oof_ridge_word_text_join_clean_real_weight_log
225794,2.307511,3.127332,2.848040,0.489404
127330,2.514768,3.283234,2.998734,0.549621
76364,2.284765,3.168813,2.940735,0.315199
79315,2.503339,3.516934,3.250256,0.737648
285898,2.525519,3.446303,3.238709,0.498193


In [95]:
feat_oof_ridge_word_text_join = train_oof_word_text_join_log.columns.to_list()

#### 4.4 Ridge на RuBert эмбеддингах text_join (title + description)

In [96]:
# чтение данных
train_rubert_emb = pd.read_parquet("RuBert_train.parquet")
test_rubert_emb = pd.read_parquet("RuBert_test.parquet")

In [97]:
sub_tr = train[["item_id"] + targets]
sub_v = val[["item_id"] + targets]

train_rubert = pd.merge(sub_tr, train_rubert_emb, how="left", on="item_id").reset_index(drop=True)
val_rubert = pd.merge(sub_v, train_rubert_emb, how="left", on="item_id").reset_index(drop=True)
test_rubert = test_rubert_emb.reset_index(drop=True)

# приведем в соответствие индексы
train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

rubert_emb_cols = train_rubert_emb.columns[1:]

In [98]:
def oof_ridge_on_emb_multi_log(
    emb_name: str,
    train_df: pd.DataFrame,
    holdout_df: pd.DataFrame,
    test_df: pd.DataFrame,
    emb_cols: list[str],
    targets: list[str],
    n_splits: int = 5,
    seed: int = 42,
    alpha: float = 10.0,
):
    X_train = train_df[emb_cols].values
    X_hold  = holdout_df[emb_cols].values
    X_test  = test_df[emb_cols].values

    # OOF / holdout / test
    oof_log = {t: np.zeros(len(train_df), dtype=np.float32) for t in targets}
    hold_log_sum = {t: np.zeros(len(holdout_df), dtype=np.float32) for t in targets}
    test_log_sum = {t: np.zeros(len(test_df), dtype=np.float32) for t in targets}

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
        X_tr_raw = X_train[tr_idx]
        X_val_raw = X_train[val_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr_raw)
        X_val = scaler.transform(X_val_raw)
        X_h   = scaler.transform(X_hold)
        X_te  = scaler.transform(X_test)

        for t in targets:
            y = train_df[t].values
            y_tr_log = np.log1p(y[tr_idx])

            model = Ridge(alpha=alpha)
            model.fit(X_tr, y_tr_log)

            # OOF (train)
            oof_log[t][val_idx] = model.predict(X_val).astype(np.float32)

            # holdout (val) - среднее по фолдам
            hold_log_sum[t] += model.predict(X_h).astype(np.float32)

            # test - среднее по фолдам
            test_log_sum[t] += model.predict(X_te).astype(np.float32)

        print(f"fold {fold}/{n_splits} done")

    # OOF dataframe
    oof_df = pd.DataFrame(
        {f"oof_ridge_{emb_name}_{t}_log": oof_log[t] for t in targets},
        index=train_df.index,
    )

    holdout_pred_df = pd.DataFrame(
        {f"oof_ridge_{emb_name}_{t}_log": (hold_log_sum[t] / n_splits) for t in targets},
        index=holdout_df.index,
    )

    test_pred_df = pd.DataFrame(
        {f"oof_ridge_{emb_name}_{t}_log": (test_log_sum[t] / n_splits) for t in targets},
        index=test_df.index,
    )

    # log MAE
    scores = []
    for t in targets:
        y_true_log = np.log1p(train_df[t].values)
        y_pred_log = oof_log[t]
        mae_log = mean_absolute_error(y_true_log, y_pred_log)
        scores.append({"target": t, "mae_log": mae_log})

    scores_df = pd.DataFrame(scores)
    scores_df.loc["mean"] = ["mean", scores_df["mae_log"].mean()]

    return oof_df, holdout_pred_df, test_pred_df, scores_df


train_oof_rubert_emb_log, val_oof_rubert_emb_log, test_oof_rubert_emb_log, scores_rubert = (
    oof_ridge_on_emb_multi_log(
        emb_name="rubert",
        train_df=train_rubert,
        holdout_df=val_rubert,
        test_df=test_rubert,
        emb_cols=rubert_emb_cols.tolist(),
        targets=targets,
        n_splits=5,
        seed=42,
        alpha=10.0,
    )
)

train = pd.concat([train, train_oof_rubert_emb_log], axis=1).copy()
val   = pd.concat([val,   val_oof_rubert_emb_log],   axis=1).copy()
test  = pd.concat([test,  test_oof_rubert_emb_log],  axis=1).copy()

display(scores_rubert)
train_oof_rubert_emb_log.head()

/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.58406e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.58406e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.58406e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.58406e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


fold 1/5 done


/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.58419e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.58419e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.58419e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.58419e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


fold 2/5 done


/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.57647e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.57647e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.57647e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.57647e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


...

fold 4/5 done


/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.59672e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.59672e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.59672e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/Users/maksimdegtarev/Documents/AAA/vs_code_pr/.venv/lib/python3.12/site-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.59672e-08): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


fold 5/5 done


,target,mae_log
0,real_height,0.363689
1,real_length,0.338276
2,real_width,0.334803
3,real_weight,0.379471
mean,mean,0.354060


,oof_ridge_rubert_real_height_log,oof_ridge_rubert_real_length_log,oof_ridge_rubert_real_width_log,oof_ridge_rubert_real_weight_log
0,2.561164,3.142619,2.923748,0.570866
1,2.611091,3.274780,2.973803,0.709314
2,2.477131,3.476799,3.146960,0.701998
3,2.613830,3.552629,3.237770,0.868401
4,2.481680,3.497502,3.163499,0.505008


In [99]:
feat_oof_ridge_rubert_text_join = train_oof_rubert_emb_log.columns.to_list()

#### 4.5 Ridge на Dino-v2 эмбеддингах фотографий

In [100]:
# чтение данных

train_dino_emb = pd.read_parquet("dinov2_train.parquet")
test_dino_emb = pd.read_parquet("dinov2_test.parquet")

In [101]:
dino_emb_cols = train_dino_emb.columns[1:]

In [102]:
sub_tr = train[targets + ["image_name"]]
sub_v = val[targets + ["image_name"]]
train_dino = pd.merge(sub_tr, train_dino_emb, how="left", on="image_name")
val_dino = pd.merge(sub_v, train_dino_emb, how="left", on="image_name")

In [103]:
train_oof_dino_emb_log, val_oof_dino_emb_log, test_oof_dino_emb_log, scores_dino = oof_ridge_on_emb_multi_log(
    emb_name="dino",
    train_df=train_dino,
    holdout_df=val_dino,
    test_df=test_dino_emb,
    emb_cols=dino_emb_cols.tolist(),
    targets=targets,
    n_splits=5,
    seed=42,
    alpha=10.0,
)

train = pd.concat([train, train_oof_dino_emb_log], axis=1).copy()
val = pd.concat([val, val_oof_dino_emb_log], axis=1).copy()
test = pd.concat([test, test_oof_dino_emb_log], axis=1).copy()

display(scores_dino)
train_oof_dino_emb_log.head()

fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done


,target,mae_log
0,real_height,0.329212
1,real_length,0.295195
2,real_width,0.298471
3,real_weight,0.308438
mean,mean,0.307829


,oof_ridge_dino_real_height_log,oof_ridge_dino_real_length_log,oof_ridge_dino_real_width_log,oof_ridge_dino_real_weight_log
0,2.426170,3.091964,2.794415,0.445690
1,2.057183,2.947296,2.818135,0.254577
2,2.316590,3.111253,2.824598,0.403397
3,2.686942,3.639056,3.347269,0.793225
4,2.770596,3.543387,3.333820,0.770053


In [104]:
feat_oof_ridge_dino = train_oof_dino_emb_log.columns.to_list()

### 5. Обучение итоговой GB модели

In [105]:
# списки признаков для модели
cat_features = ["item_condition", "category_name",
                "subcategory_name", "microcat_name"] + feat_extra_cat

meta_features = (feat_oof_ridge_tfidf_title_clean
                 + feat_oof_ridge_word_text_join
                 + feat_oof_ridge_word_title_clean
                 + feat_oof_ridge_rubert_text_join
                 + feat_oof_ridge_dino
                 + feat_meta_proxy) 

feature_cols = cat_features + meta_features + feat_extra_num

targets = [
    "real_weight",
    "real_height",
    "real_width",
    "real_length"
]

In [106]:
# проверка что все фичи не содержат Nan
assert train[feature_cols].isna().sum().sum() == 0
assert val[feature_cols].isna().sum().sum() == 0
assert test[feature_cols].isna().sum().sum() == 0

#### 5.1 Подберем гиперпараметры для итоговой модели, используя random search

In [ ]:
def eval_params(
        params: dict,
        train_df: pd.DataFrame,
        val_df: pd.DataFrame,
        feature_cols: list[str],
        cat_features: list[str],
        targets: list[str]
) -> float:
    """
    Функция для оценки параметров CatBoostRegressor
    """
    scores = []
    for t in targets:
        train_pool = Pool(
            train_df[feature_cols],
            label=np.log1p(train_df[t]),
            cat_features=cat_features,
        )
        val_pool = Pool(
            val_df[feature_cols],
            label=np.log1p(val_df[t]),
            cat_features=cat_features,
        )

        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)

        z_pred = model.predict(val_pool)
        z_true = np.log1p(val_df[t]).to_numpy()
        scores.append(np.mean(np.abs(z_true - z_pred)))

    return float(np.mean(scores))


def sample_params(rng: np.random.Generator) -> dict:
    """
    Рандомный сэмплер гиперпараметров для CatBoost
    """
    # глубина деревьев
    depth = int(rng.integers(6, 11)) # 6..10

    # логарифмический сэмплинг lr
    lr = float(10 ** rng.uniform(-2.2, -1.0)) # ~0.006..0.1

    # L2-регуляризация листьев
    l2 = float(10 ** rng.uniform(0.0, 1.3)) # ~1..20

    # доля признаков, используемых при построении дерева
    rsm = float(rng.uniform(0.6, 1.0))

    # минимальное число объектов в листе
    min_leaf = int(rng.integers(1, 80))

    if rng.random() < 0.5:
        params = dict(
            bootstrap_type="Bernoulli",
            subsample=float(rng.uniform(0.6, 1.0)),
        )
    else:
        params = dict(
            bootstrap_type="Bayesian",
            bagging_temperature=float(rng.uniform(0.0, 1.0)),
        )

    params.update(dict(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=1000,
        learning_rate=lr,
        depth=depth,
        l2_leaf_reg=l2,
        rsm=rsm,
        min_data_in_leaf=min_leaf,
        random_strength=float(rng.uniform(0.0, 2.0)), # коэф. шума, который добавляется к оценке качества сплитов при построении дерева.
        random_seed=42,
        od_type="Iter",
        od_wait=300,
        verbose=False,
        thread_count=4,
    ))

    return params


# Random search
rng = np.random.default_rng(42)

N_TRIALS = 40
best_score = 1e9
best_params = None

for i in trange(N_TRIALS, desc="Random search", unit="trial"):
    params_i = sample_params(rng)
    sc = eval_params(
        params_i,
        train,
        val,
        feature_cols,
        cat_features,
        targets,
    )

    if sc < best_score:
        best_score, best_params = sc, params_i
        print(f"\n[{i+1}/{N_TRIALS}] new best score={best_score:.6f}")
    else:
        print(f"\n[{i+1}/{N_TRIALS}] score={sc:.6f}")

print("BEST:", best_score)
print(best_params)


#### 5.2 Итоговое обучение модели на всем train на лучших параметрах из random search

In [107]:
best_params = {'bootstrap_type': 'Bernoulli',
        'subsample': 0.8607724102319897,
        'loss_function': 'MAE',
        'eval_metric': 'MAE',
        'iterations': 1000,
        'learning_rate': 0.033398625096323666,
        'depth': 10,
        'l2_leaf_reg': 3.438294242829107,
        'rsm': 0.7496736173628731,
        'min_data_in_leaf': 69,
        'random_strength': 1.7349812635046498,
        'random_seed': 42,
        'od_type': 'Iter',
        'od_wait': 300,
        'verbose': False,
        'thread_count': 4}

In [ ]:
params = best_params.copy()
params.update(dict(
        iterations=1600,
        verbose=200,
        thread_count=-1
))

models = {}
scores = {}

for t in targets:

    train_pool = Pool(
        data=train[feature_cols],
        label=np.log1p(train[t]),
        cat_features=cat_features,
    )

    val_pool = Pool(
        data=val[feature_cols],
        label=np.log1p(val[t]),
        cat_features=cat_features,
    )

    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)

    z_pred = model.predict(val_pool)
    z_true = np.log1p(val[t]).to_numpy()
    log_mae = float(np.mean(np.abs(z_true - z_pred)))

    models[t] = model
    scores[t] = log_mae

    print(f"{t}: Val Log-MAE = {log_mae:.6f} | bestIter = {model.get_best_iteration()}")

score = float(np.mean(list(scores.values())))
print(f"score = {score}")

0:	learn: 0.3969619	test: 0.4013242	best: 0.4013242 (0)	total: 329ms	remaining: 8m 46s
200:	learn: 0.2304051	test: 0.2380880	best: 0.2380880 (200)	total: 36.8s	remaining: 4m 16s
400:	learn: 0.2230173	test: 0.2342985	best: 0.2342985 (400)	total: 1m 13s	remaining: 3m 40s
600:	learn: 0.2171240	test: 0.2323503	best: 0.2323503 (600)	total: 1m 48s	remaining: 3m 1s
800:	learn: 0.2119999	test: 0.2313419	best: 0.2313414 (799)	total: 2m 28s	remaining: 2m 27s
1000:	learn: 0.2082886	test: 0.2308068	best: 0.2308068 (1000)	total: 3m 5s	remaining: 1m 51s
1200:	learn: 0.2053275	test: 0.2304233	best: 0.2304197 (1199)	total: 3m 42s	remaining: 1m 13s
1400:	learn: 0.2027630	test: 0.2301395	best: 0.2301368 (1399)	total: 4m 16s	remaining: 36.5s
1599:	learn: 0.2007172	test: 0.2299413	best: 0.2299413 (1599)	total: 4m 51s	remaining: 0us

bestTest = 0.2299412799
bestIteration = 1599

real_weight: Val Log-MAE = 0.229942 | bestIter = 1599
0:	learn: 0.3837830	test: 0.5945828	best: 0.5945828 (0)	total: 222ms	remain

#### Самая большая ошибка на val датасете на real_height, вероятно дело в заглушках. Обучим итоговую модель на всем трейне

In [109]:
full_train = pd.concat([train, val], axis=0)

In [110]:
models = {}

for t in targets:
    train_pool = Pool(
        data=full_train[feature_cols],
        label=np.log1p(full_train[t]),
        cat_features=cat_features,
    )
    
    params_final = params.copy()
    params_final.pop("od_type", None)
    params_final.pop("od_wait", None)

    model = CatBoostRegressor(**params_final)
    model.fit(train_pool, verbose=200)

    models[t] = model

print("Final models trained on full train")

0:	learn: 0.3978633	total: 272ms	remaining: 7m 15s
200:	learn: 0.2309191	total: 34.2s	remaining: 3m 57s
400:	learn: 0.2238660	total: 1m 14s	remaining: 3m 43s
600:	learn: 0.2180798	total: 1m 55s	remaining: 3m 12s
800:	learn: 0.2137009	total: 2m 39s	remaining: 2m 39s
1000:	learn: 0.2103153	total: 3m 22s	remaining: 2m 1s
1200:	learn: 0.2073845	total: 4m 7s	remaining: 1m 22s
1400:	learn: 0.2050019	total: 4m 50s	remaining: 41.2s
1599:	learn: 0.2030608	total: 5m 33s	remaining: 0us
0:	learn: 0.4046669	total: 179ms	remaining: 4m 46s
200:	learn: 0.3142224	total: 41.4s	remaining: 4m 48s
400:	learn: 0.3084708	total: 1m 26s	remaining: 4m 19s
600:	learn: 0.3030955	total: 2m 11s	remaining: 3m 38s
800:	learn: 0.2994376	total: 3m 5s	remaining: 3m 4s
1000:	learn: 0.2965229	total: 3m 51s	remaining: 2m 18s
1200:	learn: 0.2942073	total: 4m 43s	remaining: 1m 34s
1400:	learn: 0.2925341	total: 5m 36s	remaining: 47.7s
1599:	learn: 0.2911148	total: 6m 21s	remaining: 0us
0:	learn: 0.3504152	total: 256ms	remaini

In [111]:
# pool для теста
test_pool = Pool(
    data=test[feature_cols],
    cat_features=cat_features,
)

# предсказания
preds = {}
for t in targets:
    z_pred = models[t].predict(test_pool)
    y_pred = np.expm1(z_pred)
    preds[t] = y_pred.astype(float)

submission = pd.DataFrame({
    "item_id": test["item_id"].values,
    "weight": preds["real_weight"],
    "height": preds["real_height"],
    "length": preds["real_length"],
    "width": preds["real_width"]
})

submission.to_csv("submission.csv", index=False)